In [12]:
import os
# Disable MKL (major cause of memory aborts on Windows + CPU)
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"


In [14]:
import tensorflow as tf
from tensorflow.keras.models import load_model, Model
from tensorflow.keras import layers
import numpy as np
import matplotlib.pyplot as plt
from glob import glob
from sklearn.metrics import classification_report


In [1]:
BATCH_SIZE = 4          # 🔴 CRITICAL: reduce memory usage
IMG_SIZE = (224, 224)
AUTOTUNE = tf.data.AUTOTUNE
DATA_DIR = "./dataset_new"
NUM_MODELS = 5


NameError: name 'tf' is not defined

In [17]:
# -----------------------------
# DATA PREPARATION
# -----------------------------
classes = sorted(os.listdir(DATA_DIR))
class_to_index = {c: i for i, c in enumerate(classes)}

image_paths, labels = [], []
for cls in classes:
    paths = glob(os.path.join(DATA_DIR, cls, "*"))
    image_paths.extend(paths)
    labels.extend([class_to_index[cls]] * len(paths))

image_paths = np.array(image_paths)
labels = np.array(labels)

print("Classes:", classes)
print("Dataset size:", len(image_paths))

Classes: ['normal', 'osteoporosis']
Dataset size: 1945


In [18]:
def preprocess(path, label, augment=False):
    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=1, expand_animations=False)
    img = tf.image.resize(img, IMG_SIZE)
    img = tf.cast(img, tf.float32) / 255.0
    img = tf.tile(img, [1, 1, 3])

    if augment:
        img = tf.image.random_flip_left_right(img)
        img = tf.image.random_brightness(img, 0.1)
        img = tf.image.random_contrast(img, 0.9, 1.1)

    return img, label


In [19]:
dataset = tf.data.Dataset.from_tensor_slices((image_paths, labels))
dataset = dataset.shuffle(len(image_paths), seed=42)

train_size = int(0.5 * len(image_paths))
val_size = int(0.25 * len(image_paths))

train_ds = dataset.take(train_size)
val_test = dataset.skip(train_size)
val_ds = val_test.take(val_size)
test_ds = val_test.skip(val_size)

train_ds = train_ds.map(
    lambda x, y: preprocess(x, y, augment=True),
    num_parallel_calls=AUTOTUNE
).batch(BATCH_SIZE).prefetch(AUTOTUNE)

val_ds = val_ds.map(
    preprocess,
    num_parallel_calls=AUTOTUNE
).batch(BATCH_SIZE).prefetch(AUTOTUNE)

test_ds = test_ds.map(
    preprocess,
    num_parallel_calls=AUTOTUNE
).batch(BATCH_SIZE).prefetch(AUTOTUNE)


In [20]:
models = [
    load_model("cnn_model_0.h5"),
    load_model("cnn_model_1.h5"),
    load_model("cnn_model_2.h5"),
    load_model("cnn_model_3.h5"),
    load_model("cnn_model_4.h5"),
]

for m in models:
    m.trainable = False  # IMPORTANT for inference & stability


In [21]:
weights = np.array([1.0, 1.0, 1.2, 1.0, 1.5])
weights = weights / weights.sum()

In [22]:
def build_streamlit_safe_ensemble(models, weights):
    input_layer = layers.Input(shape=models[0].input_shape[1:])

    outputs = []
    for i, model in enumerate(models):
        out = model(input_layer)
        out = layers.Lambda(lambda x, w=weights[i]: x * w)(out)
        outputs.append(out)

    avg_output = layers.Add()(outputs)

    ensemble_model = Model(
        inputs=input_layer,
        outputs=avg_output,
        name="custom_cnn_ensemble"
    )

    return ensemble_model


In [23]:
ensemble_model = build_streamlit_safe_ensemble(models, weights)

ensemble_model.save("final_custom_cnn_ensemble.h5")
print("✅ Ensemble model saved successfully!")

✅ Ensemble model saved successfully!


In [24]:
# -----------------------------
# EVALUATION FUNCTION
# -----------------------------
def evaluate_ensemble(model, dataset, thresholds=None):
    """
    thresholds: only used for binary classification with sigmoid
    """
    y_true, y_pred = [], []
    num_classes = model.output_shape[-1]

    for imgs, labels in dataset:
        probs = model.predict(imgs)
        if num_classes == 1:
            # binary classification
            if thresholds is None:
                threshold = 0.5
            else:
                threshold = thresholds
            preds = (probs > threshold).astype(int).flatten()
        else:
            # multi-class
            preds = np.argmax(probs, axis=1)

        y_true.extend(labels.numpy())
        y_pred.extend(preds)

    print("\nClassification Report:")
    print(classification_report(y_true, y_pred))

In [25]:
# -----------------------------
# RUN EVALUATION
# -----------------------------
# If binary, test multiple thresholds
thresholds = [0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7]
num_classes = ensemble_model.output_shape[-1]

if num_classes == 1:
    for t in thresholds:
        print(f"\n--- Threshold: {t} ---")
        evaluate_ensemble(ensemble_model, test_ds, thresholds=t)
else:
    evaluate_ensemble(ensemble_model, test_ds)


--- Threshold: 0.3 ---
1/1 ━━━━━━━━━━━━━━━━━━━━ 7s 7s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
1/1 

AbortedError: Graph execution error:

Detected at node custom_cnn_ensemble_1/sequential_8_1/conv2d_52_1/Relu defined at (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main

  File "<frozen runpy>", line 88, in _run_code

  File "C:\Users\jaina\AppData\Roaming\Python\Python311\site-packages\ipykernel_launcher.py", line 18, in <module>

  File "C:\Users\jaina\AppData\Roaming\Python\Python311\site-packages\traitlets\config\application.py", line 1075, in launch_instance

  File "C:\Users\jaina\AppData\Roaming\Python\Python311\site-packages\ipykernel\kernelapp.py", line 739, in start

  File "C:\Users\jaina\AppData\Roaming\Python\Python311\site-packages\tornado\platform\asyncio.py", line 211, in start

  File "c:\Program Files\Python311\Lib\asyncio\base_events.py", line 608, in run_forever

  File "c:\Program Files\Python311\Lib\asyncio\base_events.py", line 1936, in _run_once

  File "c:\Program Files\Python311\Lib\asyncio\events.py", line 84, in _run

  File